# Step 1: Import all necessary libraries

In [150]:
# Libraries
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
from kaleido import get_chrome_sync

# Traffic
from traffic.core import Traffic
from traffic.data import airports, opensky
from traffic.algorithms.ground.graphs import AirportGraph
from traffic.algorithms.ground.kalman_taxiway import KalmanTaxiway
from traffic.data import aircraft

# Geospatial (preprocessing)
import cartopy.crs as ccrs
import shapely
from shapely.ops import unary_union
from shapely.geometry import (
    Point,
    LineString,
    MultiLineString,
    Polygon,
    MultiPolygon,
    GeometryCollection,
)

## Step 2: Import existing bottleneck features

In [151]:
helper_bottleneck_features = pd.read_pickle('data/processed/helper_bottleneck_features.pkl')

In [152]:
helper_bottleneck_features.head()

,flight_id,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,sum_seg_length_m,n_intersections_passed,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34
0,AAL93_1089,174.2,243.0,634.2,1401.5,8,1,0,0,0,0
1,ACA881_1093,105.6,172.5,618.4,990.5,5,1,0,0,0,0
2,AEA85MU_045,69.9,107.0,302.6,592.2,4,1,0,0,0,0
3,AEA97WK_032,185.7,398.5,945.8,1732.6,9,1,0,0,0,0
4,AEE5ZH_222,96.3,151.0,484.9,858.7,5,1,0,0,0,0


## Step 3: Import processed trajectories

In [153]:
trajs_processed = pd.read_pickle('data/processed/trajs_processed.pkl')

In [154]:
trajs_processed.head()

,timestamp,icao24,latitude,longitude,groundspeed,track,vertical_rate,callsign,onground,alert,spi,squawk,altitude,geoaltitude,lastcontact,serials,hour,flight_id,registration,typecode,track_unwrapped,cumdist,compute_gs,compute_track,cut_runway,x,y,groundspeed_xy,track_xy,latitude_kalman,longitude_kalman,groundspeed_kalman,track_kalman
0,2025-01-01 17:12:05+00:00,3e377e,47.456238,8.570239,<NA>,NaN,<NA>,DIMFE,True,False,False,3010,<NA>,<NA>,1735751524.511,"[1996109518, -1408231728]",2025-01-01 17:00:00+00:00,DIMFE_140,D-IMFE,C25A,NaN,NaN,6.881205,6.391696,28,1672.712085,-201.880485,6.664714,6.375349,47.456270,8.570223,5.523060,6.101846
1,2025-01-01 17:12:06+00:00,3e377e,47.456269,8.570244,<NA>,NaN,<NA>,DIMFE,True,False,False,3010,<NA>,<NA>,1735751525.835,"[1996109518, -1408231728]",2025-01-01 17:00:00+00:00,DIMFE_140,D-IMFE,C25A,NaN,NaN,6.881205,6.391696,28,1673.092805,-198.473063,6.664714,6.375345,47.456293,8.570227,5.654918,6.068220
2,2025-01-01 17:12:07+00:00,3e377e,47.456299,8.570249,<NA>,NaN,<NA>,DIMFE,True,False,False,3010,<NA>,<NA>,1735751525.835,"[1996109518, -1408231728]",2025-01-01 17:00:00+00:00,DIMFE_140,D-IMFE,C25A,NaN,NaN,6.881205,6.391693,28,1673.473524,-195.065641,6.664714,6.375342,47.456319,8.570231,5.744058,6.002628
3,2025-01-01 17:12:08+00:00,3e377e,47.45633,8.570254,<NA>,NaN,<NA>,DIMFE,True,False,False,3010,<NA>,<NA>,1735751525.835,"[1996109518, -1408231728]",2025-01-01 17:00:00+00:00,DIMFE_140,D-IMFE,C25A,NaN,NaN,6.881205,6.391689,28,1673.854242,-191.658219,6.664714,6.375334,47.456347,8.570235,5.859442,5.886520
4,2025-01-01 17:12:09+00:00,3e377e,47.456361,8.570259,<NA>,NaN,<NA>,DIMFE,True,False,False,3010,<NA>,<NA>,1735751525.835,"[1996109518, -1408231728]",2025-01-01 17:00:00+00:00,DIMFE_140,D-IMFE,C25A,NaN,NaN,6.881205,6.391685,28,1674.234961,-188.250797,6.664714,6.375327,47.456376,8.570240,5.895811,5.745141


# Step 4: Import processed meteorological data

In [155]:
meteo_processed = pd.read_pickle('data/processed/meteo_processed.pkl')

## Step 5: Feature Engineering

In the following steps, additional features based on the processed trajectories are being created and then merged with the existing bottleneck features.

### Step 5.1: Operational Features

First, we create a features dataframe and define the target variable (taxi out time) for each flight_id.

In [156]:
df = trajs_processed

features_df = (
    df.groupby("flight_id", sort=False)
      .agg(
          taxi_start=("timestamp", "min"),
          taxi_end=("timestamp", "max"),
      )
      .reset_index()
)

features_df["taxi_time_s"] = (
    features_df["taxi_end"]
    - features_df["taxi_start"]
).dt.total_seconds()

In [157]:
features_df.head()

,flight_id,taxi_start,taxi_end,taxi_time_s
0,DIMFE_140,2025-01-01 17:12:05+00:00,2025-01-01 17:12:12+00:00,7.0
1,CTN465_1006,2025-01-01 19:52:20+00:00,2025-01-01 19:53:18+00:00,58.0
2,AUA570_199,2025-01-01 14:16:54+00:00,2025-01-01 14:18:41+00:00,107.0
3,ASL83N_212,2025-01-01 22:04:57+00:00,2025-01-01 22:06:56+00:00,119.0
4,CTN461_1011,2025-01-01 10:05:23+00:00,2025-01-01 10:07:37+00:00,134.0


Before defining a filtering threshold, the distribution of taxi-out times is inspected using summary statistics and lower quantiles to understand the realistic range of values.

In [158]:
def inspect_taxi_time_distribution(features_df):

    print("Summary statistics:")
    print(features_df["taxi_time_s"].describe())

    print("\nLower quantiles:")
    print(
        features_df["taxi_time_s"].quantile(
            [0.01, 0.02, 0.05, 0.10]
        )
    )


inspect_taxi_time_distribution(features_df)

Summary statistics:
count         287.0
mean     644.515679
std      432.661885
min             3.0
25%           365.5
50%           583.0
75%           785.5
max          2378.0
Name: taxi_time_s, dtype: double[pyarrow]

Lower quantiles:
0.01    10.44
0.02    48.16
0.05    107.3
0.10    135.8
Name: taxi_time_s, dtype: double[pyarrow]


A configurable minimum taxi-out time parameter is defined to allow flexible adjustment of the filtering threshold as the dataset grows or data characteristics change.


 <font color='red'> CHANGE THE PARAMETER IN THE CELL BELOW: </font>

In [159]:
MIN_TAXI_TIME_S = 60

Flights with taxi-out times below the defined minimum threshold are excluded from further analysis to remove incomplete or operationally implausible trajectories.

In [160]:
features_df = features_df[
    features_df["taxi_time_s"] >= MIN_TAXI_TIME_S
]

Adding aircraft typecode as a feature and setting datatype as "category"

In [161]:
aircraft_lookup = (
    trajs_processed[["icao24"]]
    .drop_duplicates()
    .assign(
        typecode=lambda df: df["icao24"].apply(
            lambda x: aircraft.get(x).typecode
            if aircraft.get(x) is not None
            else None
        )
    )
)

flight_aircraft = (
    trajs_processed[["flight_id", "icao24"]]
    .drop_duplicates()
    .merge(aircraft_lookup, on="icao24", how="left")
)

features_df = features_df.merge(
    flight_aircraft[["flight_id", "typecode"]],
    on="flight_id",
    how="left"
)

features_df["typecode"] = features_df["typecode"].astype("category")

The queue size feature is calculated within a rolling one-hour lookback window. For each flight, only aircraft that started taxiing within the previous hour and were still taxiing at the current flight's taxi_start time are counted. This keeps the feature operationally realistic, prediction-safe, and computationally scalable for larger datasets.

In [162]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]

queue_size_last_hour = []

for i, t in enumerate(starts):
    n_active = (
        (starts >= t - WINDOW)
        & (starts < t)
        & (ends > t)
    ).sum()

    queue_size_last_hour.append(n_active)

features_df["queue_size"] = queue_size_last_hour

Implementing the average taxi out time of the past 6 hours as a feature

In [163]:
WINDOW = pd.Timedelta(hours=6)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_6h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_6h.append(avg)

features_df["avg_taxi_out_time_6h"] = avg_taxi_out_6h

Implementing the average taxi out time of the past hour as a feature.

In [164]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_1h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_1h.append(avg)

features_df["avg_taxi_out_time_1h"] = avg_taxi_out_1h

Implementing the average taxi out time of the past 15 minutes as feature.

In [165]:
WINDOW = pd.Timedelta(hours=0.25)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_15min = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_15min.append(avg)

features_df["avg_taxi_out_time_15min"] = avg_taxi_out_15min

Implementing bottleneck features

In [166]:
cols_to_merge = [
    "flight_id",
    "n_intersections_passed",
    "sum_seg_length_m",
    "TO_runway_28",
    "TO_runway_32",
    "TO_runway_16",
    "TO_runway_10",
    "TO_runway_34",
]

features_df = features_df.merge(
    helper_bottleneck_features[cols_to_merge],
    on="flight_id",
    how="left",
)

### Step 5.2 Temporal Features

Defining a function for cyclic encoding

In [167]:
def sin_cos(x,T):
    return np.sin(2 * np.pi * x / T), np.cos(2 * np.pi * x / T)

Splitting up information from the 'taxi_start' column into month, day of week and minute of day

In [168]:
features_df["month"] = features_df["taxi_start"].dt.month

features_df["day_of_week"] = features_df["taxi_start"].dt.dayofweek
# Monday = 0, Sunday = 6

features_df["minute_of_day"] = (
    features_df["taxi_start"].dt.hour * 60
    + features_df["taxi_start"].dt.minute
)

Encoding month, day of week and minute of day as sin and cos

In [169]:
features_df["month_sin"], features_df["month_cos"] = sin_cos(
    features_df["month"], 12
)

features_df["day_of_week_sin"], features_df["day_of_week_cos"] = sin_cos(
    features_df["day_of_week"], 7
)

features_df["minute_of_day_sin"], features_df["minute_of_day_cos"] = sin_cos(
    features_df["minute_of_day"], 24 * 60
)

### Step 5.3: Meteorological Features

Implementing temperature as a feature

In [170]:
meteo_temp = (
    meteo_processed[
        ["reference_timestamp", "tre200s0"]
    ]
    .rename(
        columns={
            "tre200s0": "temp_at_taxi_start"
        }
    )
)

features_df = pd.merge_asof(
    features_df.sort_values("taxi_start"),
    meteo_temp.sort_values("reference_timestamp"),
    left_on="taxi_start",
    right_on="reference_timestamp",
    direction="backward"
)

features_df = features_df.drop(columns="reference_timestamp")

In [171]:
features_df.head(15)

,flight_id,taxi_start,taxi_end,taxi_time_s,typecode,queue_size,avg_taxi_out_time_6h,avg_taxi_out_time_1h,avg_taxi_out_time_15min,n_intersections_passed,sum_seg_length_m,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34,month,day_of_week,minute_of_day,month_sin,month_cos,day_of_week_sin,day_of_week_cos,minute_of_day_sin,minute_of_day_cos,temp_at_taxi_start
0,GSW444_577,2025-01-01 05:34:37+00:00,2025-01-01 05:37:14+00:00,157.0,A320,0,NaN,NaN,NaN,3,556.0,0,1,0,0,0,1,2,334,0.5,0.866025,0.974928,-0.222521,0.993572,0.113203,-2.5
1,THY8MQ_937,2025-01-01 05:43:08+00:00,2025-01-01 06:13:11+00:00,1803.0,B739,0,157.000000,157.000000,157.000000,7,1301.9,1,0,0,0,0,1,2,343,0.5,0.866025,0.974928,-0.222521,0.997250,0.074108,-2.7
2,EDW402_523,2025-01-01 05:56:41+00:00,2025-01-01 06:19:41+00:00,1380.0,A320,1,157.000000,157.000000,NaN,12,1850.2,1,0,0,0,0,1,2,356,0.5,0.866025,0.974928,-0.222521,0.999848,0.017452,-2.6
3,SWR8YH_482,2025-01-01 06:02:22+00:00,2025-01-01 06:23:16+00:00,1254.0,BCS3,2,157.000000,157.000000,NaN,13,2322.2,1,0,0,0,0,1,2,362,0.5,0.866025,0.974928,-0.222521,0.999962,-0.008727,-2.9
4,AFR94BH_056,2025-01-01 06:04:22+00:00,2025-01-01 06:15:57+00:00,695.0,NaN,3,157.000000,157.000000,NaN,6,1087.0,1,0,0,0,0,1,2,364,0.5,0.866025,0.974928,-0.222521,0.999848,-0.017452,-2.9
5,SWR464L_1020,2025-01-01 06:05:14+00:00,2025-01-01 06:26:28+00:00,1274.0,BCS3,4,157.000000,157.000000,NaN,12,1880.0,1,0,0,0,0,1,2,365,0.5,0.866025,0.974928,-0.222521,0.999762,-0.021815,-2.9
6,BAW709_146,2025-01-01 06:18:54+00:00,2025-01-01 06:30:19+00:00,685.0,A319,3,885.000000,885.000000,1249.000000,7,1313.0,1,0,0,0,0,1,2,378,0.5,0.866025,0.974928,-0.222521,0.996917,-0.078459,-2.7
7,SWR37P_305,2025-01-01 06:21:22+00:00,2025-01-01 06:37:42+00:00,980.0,A320,3,1008.750000,1008.750000,1292.666667,10,1530.5,1,0,0,0,0,1,2,381,0.5,0.866025,0.974928,-0.222521,0.995805,-0.091502,-2.8
8,SWR44C_328,2025-01-01 06:22:26+00:00,2025-01-01 06:34:51+00:00,745.0,A320,4,1008.750000,1008.750000,1292.666667,6,878.7,1,0,0,0,0,1,2,382,0.5,0.866025,0.974928,-0.222521,0.995396,-0.095846,-2.8
9,SWR1GT_429,2025-01-01 06:28:36+00:00,2025-01-01 06:44:19+00:00,943.0,BCS1,3,1093.833333,1093.833333,1150.750000,8,1343.0,1,0,0,0,0,1,2,388,0.5,0.866025,0.974928,-0.222521,0.992546,-0.121869,-2.8
